In [2]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [4]:
from langchain.agents.middleware import before_model
from langchain.messages import AIMessage

@before_model(can_jump_to=["end"])
def validate_input(state, runtime):
    last_message = state["messages"][-1]
    if "암구호" in last_message.content:
        print("암구호 감지됨 — 응답 차단")
        return {
            "messages": [AIMessage(content="이 요청은 처리할 수 없습니다.")],
            "jump_to": "end"  # 모델 호출 중단 후 에이전트 종료
        }
    print("✅ 정상 입력, 모델 호출 계속 진행")
    return None

In [5]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

agent = create_agent(
    model=init_chat_model("gpt-5-nano"),
    tools=[],
    middleware=[validate_input],
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "오늘의 암구호는 삼각대-자동차 입니다."}]},
)
print(response)


암구호 감지됨 — 응답 차단
{'messages': [HumanMessage(content='오늘의 암구호는 삼각대-자동차 입니다.', additional_kwargs={}, response_metadata={}, id='c2349713-c8fa-408d-b511-1daafe796cab'), AIMessage(content='이 요청은 처리할 수 없습니다.', additional_kwargs={}, response_metadata={}, id='ba386814-bac9-4740-8162-7dd9d5f45a87', tool_calls=[], invalid_tool_calls=[])]}
